© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# HAIO 2024 - Transfer Learning Solution

This notebook contains the solution to the transfer learning task of the Hungarian Artificial Intelligence Olympiad (HAIO) 2024.

The essence of the task: transferring the knowledge of a ResNet model trained on CIFAR-100 to CIFAR-10.

**Subtasks:**
1. Before transfer learning - training and evaluating the models
2. Transfer learning - fine-tuning the last layer
3. Distance between models - L2 distance matrix
4. Further training of multiple layers
5. Training on little data

## Preparation

Imports, device setup and helper functions.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset

# Use the GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Data augmentation and normalization
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# CIFAR-10 datasets
cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# CIFAR-10 train without augmentation (for evaluation)
cifar10_train_eval = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_test)

cifar10_trainloader = DataLoader(cifar10_train, batch_size=128, shuffle=True, num_workers=2)
cifar10_testloader = DataLoader(cifar10_test, batch_size=128, shuffle=False, num_workers=2)
cifar10_train_evalloader = DataLoader(cifar10_train_eval, batch_size=128, shuffle=False, num_workers=2)

# CIFAR-100 datasets
cifar100_train = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
cifar100_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

cifar100_trainloader = DataLoader(cifar100_train, batch_size=128, shuffle=True, num_workers=2)
cifar100_testloader = DataLoader(cifar100_test, batch_size=128, shuffle=False, num_workers=2)

print(f'CIFAR-10 train: {len(cifar10_train)}, test: {len(cifar10_test)}')
print(f'CIFAR-100 train: {len(cifar100_train)}, test: {len(cifar100_test)}')

## ResNet model adapted to CIFAR

The standard ResNet-18 was designed for ImageNet (224x224 images). The CIFAR dataset consists of 32x32 pixel images,
so we use a smaller ResNet architecture optimized for CIFAR:
- Initial convolution: 3x3, stride=1 (no maxpool)
- 3 groups of residual blocks: [16, 32, 64] channels
- Global average pooling before the FC layer

In [ ]:
class BasicBlock(nn.Module):
    """Basic residual block with 2 convolutional layers."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut connection in case of a size mismatch
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class CIFARResNet(nn.Module):
    """ResNet optimized for CIFAR.
    
    3 groups of residual blocks with [16, 32, 64] channels.
    `num_blocks` blocks per group (default: 3, which corresponds to ResNet-20).
    """

    def __init__(self, num_blocks=3, num_classes=10):
        super().__init__()
        self.in_channels = 16

        # Initial convolution - 3x3, stride=1, no maxpool
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # 3 residual groups
        self.layer1 = self._make_layer(16, num_blocks, stride=1)   # 32x32
        self.layer2 = self._make_layer(32, num_blocks, stride=2)   # 16x16
        self.layer3 = self._make_layer(64, num_blocks, stride=2)   # 8x8

        # Global average pooling and classifier layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

        # Weight initialization
        self._initialize_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


# Test: model creation and summary
test_model = CIFARResNet(num_blocks=3, num_classes=10)
total_params = sum(p.numel() for p in test_model.parameters())
print(f'Number of parameters of CIFAR ResNet-20: {total_params:,}')
del test_model

In [ ]:
def evaluate(model, dataloader, device):
    """Evaluating the model: computing accuracy."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


def train_model(model, trainloader, testloader, epochs, lr=0.1, device='cuda',
                extra_testloader=None, extra_test_name=None):
    """Training the model with an SGD optimizer and a cosine annealing scheduler.
    
    Returns the history of training and test accuracy/loss.
    If extra_testloader is provided, it is also evaluated on it in each epoch.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {
        'train_loss': [], 'train_acc': [],
        'test_acc': []
    }
    if extra_testloader is not None:
        history['extra_test_acc'] = []

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        test_acc = evaluate(model, testloader, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        log_str = f'Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% | Test acc: {test_acc:.2f}%'

        if extra_testloader is not None:
            extra_acc = evaluate(model, extra_testloader, device)
            history['extra_test_acc'].append(extra_acc)
            log_str += f' | {extra_test_name} acc: {extra_acc:.2f}%'

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(log_str)

    return history

---
## Task 1: Before transfer learning

In this part we:
1. Train a ResNet model on CIFAR-100
2. Train a ResNet model on CIFAR-10
3. Replace the last FC layer of the CIFAR-100 model with the FC layer of the CIFAR-10 model
4. Evaluate the hybrid model on CIFAR-10

In [ ]:
# Training the CIFAR-100 model
print('=== Training the CIFAR-100 model ===')
model_cifar100 = CIFARResNet(num_blocks=3, num_classes=100)
history_cifar100 = train_model(model_cifar100, cifar100_trainloader, cifar100_testloader,
                                epochs=50, lr=0.1, device=device)

cifar100_test_acc = evaluate(model_cifar100, cifar100_testloader, device)
print(f'\nFinal test accuracy of the CIFAR-100 model: {cifar100_test_acc:.2f}%')

In [ ]:
# Training the CIFAR-10 model
print('=== Training the CIFAR-10 model ===')
model_cifar10 = CIFARResNet(num_blocks=3, num_classes=10)
history_cifar10 = train_model(model_cifar10, cifar10_trainloader, cifar10_testloader,
                               epochs=50, lr=0.1, device=device)

cifar10_test_acc = evaluate(model_cifar10, cifar10_testloader, device)
print(f'\nFinal test accuracy of the CIFAR-10 model: {cifar10_test_acc:.2f}%')

In [ ]:
# Plotting the learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_cifar100['train_loss'], label='CIFAR-100 train')
axes[0].plot(history_cifar10['train_loss'], label='CIFAR-10 train')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history_cifar100['test_acc'], label='CIFAR-100 test')
axes[1].plot(history_cifar10['test_acc'], label='CIFAR-10 test')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Hybrid model: CIFAR-100 backbone + CIFAR-10 FC layer
print('=== Creating the hybrid model ===')
model_hybrid = copy.deepcopy(model_cifar100)

# Replacing the last FC layer with the FC layer of the CIFAR-10 model
model_hybrid.fc = copy.deepcopy(model_cifar10.fc)
model_hybrid = model_hybrid.to(device)

# Evaluation on CIFAR-10
hybrid_train_acc = evaluate(model_hybrid, cifar10_train_evalloader, device)
hybrid_test_acc = evaluate(model_hybrid, cifar10_testloader, device)

print(f'Hybrid model CIFAR-10 train accuracy: {hybrid_train_acc:.2f}%')
print(f'Hybrid model CIFAR-10 test accuracy: {hybrid_test_acc:.2f}%')
print(f'\nComparison:')
print(f'  Original CIFAR-10 model test acc: {cifar10_test_acc:.2f}%')
print(f'  Hybrid model test acc:            {hybrid_test_acc:.2f}%')
print(f'\nThe hybrid model performs poorly because the features learned on CIFAR-100')
print(f'and the weights of the CIFAR-10 FC layer are not aligned with each other.')

---
## Task 2: Transfer learning

We replace the last layer of the model pre-trained on CIFAR-100 with a new, randomly initialized
FC layer with 10 outputs. We freeze all layers except the last one,
then fine-tune on CIFAR-10.

Meanwhile we monitor the accuracy on both the CIFAR-10 and the CIFAR-100 test sets.

In [ ]:
# Transfer learning: we train only the last layer
print('=== Transfer learning: fine-tuning the last layer ===')

# Copying the CIFAR-100 model and replacing the last layer
model_transfer = copy.deepcopy(model_cifar100)
model_transfer.fc = nn.Linear(64, 10)  # New random FC layer with 10 outputs

# Freezing every layer except the last FC
for name, param in model_transfer.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

# Check: how many parameters we are training
trainable = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_transfer.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

# Fine-tuning
history_transfer = train_model(
    model_transfer, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.1, device=device,
    extra_testloader=cifar100_testloader,
    extra_test_name='CIFAR-100'
)

transfer_test_acc = evaluate(model_transfer, cifar10_testloader, device)
print(f'\nTransfer model CIFAR-10 test accuracy: {transfer_test_acc:.2f}%')

In [ ]:
# Plotting accuracy: CIFAR-10 and CIFAR-100 test
fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, len(history_transfer['test_acc']) + 1)
ax.plot(epochs_range, history_transfer['test_acc'], 'b-o', markersize=3,
        label='CIFAR-10 test accuracy')
ax.plot(epochs_range, history_transfer['extra_test_acc'], 'r-s', markersize=3,
        label='CIFAR-100 test accuracy')

ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Transfer learning: evolution of accuracy\n(only the last layer trained)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'Observation: The CIFAR-10 accuracy increases, while the CIFAR-100 accuracy')
print(f'stays low (since we replaced the FC layer for CIFAR-10).')
print(f'The representations learned on CIFAR-100 by the frozen feature extractor layers')
print(f'can also be used well for CIFAR-10 classification - this is the essence of transfer learning.')

---
## Task 3: Distance between models

We compute the pairwise L2 (Euclidean) distance between the parameters of 3 models:
1. Randomly initialized model (10 outputs)
2. Model trained on CIFAR-100 with the last layer replaced in Task 1 (hybrid)
3. The transfer model fine-tuned in Task 2

We compute the distance on the concatenated vector of all parameters.

In [ ]:
def get_param_vector(model):
    """Concatenating all parameters of a model into a single vector."""
    return torch.cat([p.data.view(-1) for p in model.parameters()])


def pairwise_l2_distance(models, names):
    """Computing and displaying the pairwise L2 distance matrix."""
    n = len(models)
    vectors = [get_param_vector(m).cpu() for m in models]
    dist_matrix = torch.zeros(n, n)

    for i in range(n):
        for j in range(n):
            dist_matrix[i, j] = torch.norm(vectors[i] - vectors[j], p=2).item()

    return dist_matrix


# Randomly initialized model (10 outputs, like the transfer model's)
model_random = CIFARResNet(num_blocks=3, num_classes=10).to(device)

# The CIFAR-100 model with the last layer replaced (the hybrid model of Task 1)
# We use model_hybrid (CIFAR-100 backbone + CIFAR-10 FC)
# But in line with the task we rather take the CIFAR-100 model with a new random FC
model_cifar100_swapped = copy.deepcopy(model_cifar100)
model_cifar100_swapped.fc = nn.Linear(64, 10)  # New random FC with 10 outputs
model_cifar100_swapped = model_cifar100_swapped.to(device)

# Comparing the 3 models
models = [model_random, model_cifar100_swapped, model_transfer]
names = ['Random', 'CIFAR-100 (swapped FC)', 'Fine-tuned']

dist_matrix = pairwise_l2_distance(models, names)

# Display
print('Pairwise L2 distance matrix:')
print(f'{"":>25s}', end='')
for name in names:
    print(f'{name:>25s}', end='')
print()
for i, name in enumerate(names):
    print(f'{name:>25s}', end='')
    for j in range(len(names)):
        print(f'{dist_matrix[i, j]:>25.2f}', end='')
    print()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(dist_matrix.numpy(), cmap='YlOrRd')
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_yticklabels(names)

# Displaying the values on the cells
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{dist_matrix[i, j]:.1f}',
                ha='center', va='center', fontsize=12)

ax.set_title('Pairwise L2 distance of the models')
plt.colorbar(im, ax=ax, label='L2 distance')
plt.tight_layout()
plt.show()

print(f'\nObservations:')
print(f'- The random model is far from both trained models.')
print(f'- The CIFAR-100 pre-trained and the fine-tuned model are close to each other,')
print(f'  since we only modified the last layer during fine-tuning.')

---
## Task 4: Training more layers

Now we also unfreeze more layers for fine-tuning:
- **A) From the second residual group:** layer2 + layer3 + fc
- **B) From after the second residual group:** layer3 + fc

We compare the test accuracies.

In [ ]:
def create_transfer_model(base_model, unfreeze_from='fc'):
    """Creating a transfer model unfrozen from the given layer.
    
    Args:
        base_model: model pre-trained on CIFAR-100
        unfreeze_from: 'fc', 'layer3', 'layer2', 'layer1', 'all'
    """
    model = copy.deepcopy(base_model)
    model.fc = nn.Linear(64, 10)  # New FC layer for CIFAR-10

    # By default everything is frozen
    for param in model.parameters():
        param.requires_grad = False

    # Unfreezing the layers from the given point
    layers_order = ['layer1', 'layer2', 'layer3', 'fc']
    if unfreeze_from == 'all':
        for param in model.parameters():
            param.requires_grad = True
    else:
        start_idx = layers_order.index(unfreeze_from)
        for layer_name in layers_order[start_idx:]:
            layer = getattr(model, layer_name)
            for param in layer.parameters():
                param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Unfrozen: from {unfreeze_from} | Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)')

    return model

In [ ]:
# A) Unfrozen from the second residual group (layer2 + layer3 + fc)
print('=== A) unfrozen from layer2 ===')
model_unfreeze_layer2 = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_unfreeze_layer2 = train_model(
    model_unfreeze_layer2, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_layer2 = evaluate(model_unfreeze_layer2, cifar10_testloader, device)
print(f'Final test accuracy (from layer2): {acc_layer2:.2f}%')

In [ ]:
# B) Unfrozen from the third residual group (layer3 + fc)
print('=== B) unfrozen from layer3 ===')
model_unfreeze_layer3 = create_transfer_model(model_cifar100, unfreeze_from='layer3')
history_unfreeze_layer3 = train_model(
    model_unfreeze_layer3, cifar10_trainloader, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_layer3 = evaluate(model_unfreeze_layer3, cifar10_testloader, device)
print(f'Final test accuracy (from layer3): {acc_layer3:.2f}%')

In [ ]:
# Comparison
fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, 51)
ax.plot(epochs_range, history_transfer['test_acc'], label='FC layer only', marker='.')
ax.plot(epochs_range, history_unfreeze_layer3['test_acc'], label='layer3 + FC', marker='.')
ax.plot(epochs_range, history_unfreeze_layer2['test_acc'], label='layer2 + layer3 + FC', marker='.')

ax.set_xlabel('Epoch')
ax.set_ylabel('CIFAR-10 test accuracy (%)')
ax.set_title('Effect of fine-tuning more layers')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  FC layer only:         {transfer_test_acc:.2f}%')
print(f'  layer3 + FC:           {acc_layer3:.2f}%')
print(f'  layer2 + layer3 + FC:  {acc_layer2:.2f}%')
print(f'  Original CIFAR-10:     {cifar10_test_acc:.2f}%')
print(f'\nObservation: Unfreezing more layers generally yields better accuracy,')
print(f'but with too many unfrozen layers we may lose the benefits of pre-training.')

---
## Task 5: Training on little data

We examine how transfer learning performs if we use only a portion of the data:
- 50% of the CIFAR-10 training set
- 20% of the CIFAR-10 training set

Question: can we reach the accuracy of the CIFAR-10 model trained on the full data?

In [ ]:
def create_subset_loader(dataset, fraction, batch_size=128):
    """Creating a DataLoader from a subset of a dataset."""
    n = len(dataset)
    indices = np.random.permutation(n)[:int(n * fraction)]
    subset = Subset(dataset, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2)


np.random.seed(42)  # Reproducibility

# 50% dataset
cifar10_train_50pct = create_subset_loader(cifar10_train, 0.5)
print(f'50% training set: {int(len(cifar10_train) * 0.5)} samples')

# 20% dataset
cifar10_train_20pct = create_subset_loader(cifar10_train, 0.2)
print(f'20% training set: {int(len(cifar10_train) * 0.2)} samples')

In [ ]:
# Transfer learning with 50% of the data (unfrozen from layer2, because that was the best)
print('=== Transfer learning with 50% of the data ===')
model_50pct = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_50pct = train_model(
    model_50pct, cifar10_train_50pct, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_50pct = evaluate(model_50pct, cifar10_testloader, device)
print(f'Final test accuracy (50% data): {acc_50pct:.2f}%')

In [ ]:
# Transfer learning with 20% of the data
print('=== Transfer learning with 20% of the data ===')
model_20pct = create_transfer_model(model_cifar100, unfreeze_from='layer2')
history_20pct = train_model(
    model_20pct, cifar10_train_20pct, cifar10_testloader,
    epochs=50, lr=0.01, device=device
)
acc_20pct = evaluate(model_20pct, cifar10_testloader, device)
print(f'Final test accuracy (20% data): {acc_20pct:.2f}%')

In [ ]:
# For comparison: training from scratch with limited data
print('=== Training from scratch with 50% of the data (comparison) ===')
model_scratch_50pct = CIFARResNet(num_blocks=3, num_classes=10)
history_scratch_50pct = train_model(
    model_scratch_50pct, cifar10_train_50pct, cifar10_testloader,
    epochs=50, lr=0.1, device=device
)
acc_scratch_50pct = evaluate(model_scratch_50pct, cifar10_testloader, device)
print(f'Trained from scratch (50% data) test accuracy: {acc_scratch_50pct:.2f}%')

print('\n=== Training from scratch with 20% of the data (comparison) ===')
model_scratch_20pct = CIFARResNet(num_blocks=3, num_classes=10)
history_scratch_20pct = train_model(
    model_scratch_20pct, cifar10_train_20pct, cifar10_testloader,
    epochs=50, lr=0.1, device=device
)
acc_scratch_20pct = evaluate(model_scratch_20pct, cifar10_testloader, device)
print(f'Trained from scratch (20% data) test accuracy: {acc_scratch_20pct:.2f}%')

In [ ]:
# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, 51)

# 50% data
axes[0].plot(epochs_range, history_50pct['test_acc'], label='Transfer (50%)')
axes[0].plot(epochs_range, history_scratch_50pct['test_acc'], label='From scratch (50%)', linestyle='--')
axes[0].axhline(y=cifar10_test_acc, color='green', linestyle=':', label=f'Full CIFAR-10 ({cifar10_test_acc:.1f}%)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CIFAR-10 test accuracy (%)')
axes[0].set_title('With 50% of the training data')
axes[0].legend()
axes[0].grid(True)

# 20% data
axes[1].plot(epochs_range, history_20pct['test_acc'], label='Transfer (20%)')
axes[1].plot(epochs_range, history_scratch_20pct['test_acc'], label='From scratch (20%)', linestyle='--')
axes[1].axhline(y=cifar10_test_acc, color='green', linestyle=':', label=f'Full CIFAR-10 ({cifar10_test_acc:.1f}%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('CIFAR-10 test accuracy (%)')
axes[1].set_title('With 20% of the training data')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Final summary table
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)
print(f'{"Method":<45s} {"Test acc (%)":>15s}')
print('-' * 70)
print(f'{"CIFAR-10 from scratch (100% data)":<45s} {cifar10_test_acc:>14.2f}%')
print(f'{"CIFAR-100 from scratch":<45s} {cifar100_test_acc:>14.2f}%')
print(f'{"Hybrid (CIFAR-100 backbone + CIFAR-10 FC)":<45s} {hybrid_test_acc:>14.2f}%')
print(f'{"Transfer: FC only":<45s} {transfer_test_acc:>14.2f}%')
print(f'{"Transfer: layer3 + FC":<45s} {acc_layer3:>14.2f}%')
print(f'{"Transfer: layer2 + layer3 + FC":<45s} {acc_layer2:>14.2f}%')
print(f'{"Transfer: 50% data (from layer2)":<45s} {acc_50pct:>14.2f}%')
print(f'{"From scratch: 50% data":<45s} {acc_scratch_50pct:>14.2f}%')
print(f'{"Transfer: 20% data (from layer2)":<45s} {acc_20pct:>14.2f}%')
print(f'{"From scratch: 20% data":<45s} {acc_scratch_20pct:>14.2f}%')
print('=' * 70)

print(f'\nConclusions:')
print(f'1. The features learned on CIFAR-100 transfer well to CIFAR-10.')
print(f'2. Unfreezing more layers yields better results.')
print(f'3. With transfer learning, competitive accuracy can be achieved with less data.')
print(f'4. When data is scarce, the advantage of transfer learning is even more pronounced.')